In [5]:
from dotenv import load_dotenv
load_dotenv()

True

# 预定义中间件

## PIIMiddleware-个人信息脱敏
可以在调用模型前后自动检测并脱敏输入、输出消息中的个人身份信息（PII），如邮箱、电话号码、身份证号等。

PII脱敏处理策略有四种：  
- 'block' - 抛出异常
- 'redact' - 用 [REDACTED_{PII_TYPE}] 来替代
- 'mask' - 关键信息采用**掩码 (例如., ****-****-****-1234)
- 'hash' - 用哈希值来替换

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

# 可添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  # 掩码
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="block",
    apply_to_input=True
)

# 在agent中注册
agent_with_builtin_middleware = create_agent(
    model = "deepseek-chat",
    middleware=[pii_middleware_email, pii_middleware_phone]
)

# 测试
response = agent_with_builtin_middleware.invoke(
    {"messages":[HumanMessage("我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")]}
)

for m in response["messages"]:
    m.pretty_print()

PIIDetectionError: Detected 1 instance(s) of phone_number in text content

## ModelFallbackMiddleware
模型调用失败时给出降级处理方案。可以在创建时设置多个模型，如果主模型调用失败，会自动调用备用模型。

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

# 设置主模型和备用模型
model_fallback_middleware = ModelFallbackMiddleware(
    "gpt-4o-mini",  # 默认模型
    "deepseek-chat"
)

agent_with_model_fallback = create_agent(
    model="gpt-4o-mini",
    middleware=[model_fallback_middleware]
)

# 测试
response = agent_with_model_fallback.invoke({
    "messages": [HumanMessage("你好")]
})
for m in response['messages']:
    m.pretty_print()

## HumanInTheLoopMiddleware-人工审核
让人工介入到Agent执行流程中，在执行工具调用前暂停，等待人工确认。  
通常用于Agent执行敏感操作前的确认

In [2]:
from langchain.tools import tool

@tool
def transfer_money(amount: int, to: str):
    """
    向指定账户转账
    arg:
        amount: 金额
        to: 收款人
    """
    return f"已向{to}转账{amount}元"

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={  # 需要人工确认的tool信息，可以设置多个
        "transfer_money": {  # 就是需要人工确认的tool名字
            "description": "请确认转账操作",  # 是人工确认时的提示信息
            "allowed_decisions": ["approve", 'reject', 'edit']  # 是人工确认时的可选操作
        }
    }
)

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_hitl = create_agent(
    model="deepseek-v4-flash",
    tools=[transfer_money],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个账户管理助手，你可以调用工具帮助用户转账，用户提出明确需求后，不要重复向用户确认信息。"
)

In [17]:
config = {"configurable": {"thread_id": "3"}}

response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我转2000元给王小明(6123008415124395223")]},
    config=config
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================


================================== Ai Message ==================================

您好，我是账户管理助手，可以帮助您进行转账操作。请问您需要转账给谁、转账多少金额呢？
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
================================== Ai Message ==================================
Tool Calls:
  transfer_money (call_00_FHnIlOfcPT1aEn3JTijn8930)
 Call ID: call_00_FHnIlOfcPT1aEn3JTijn8930
  Args:
    amount: 2000
    to: 王小明
================================= Tool Message =================================
Name: transfer_money

已向王小明转账2000元
================================== Ai Message ==================================

转账已完成！已成功向王小明转账2000元。请问还有其他需要帮助的吗？
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
================================== Ai Message ==================================
Tool Calls:
  transfer_

需要再次调用Agent，并通过Command来告知Agent用户的选择是什么。

approve

In [13]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================


================================== Ai Message ==================================

您好，我是账户管理助手，可以帮助您进行转账操作。请问您需要转账给谁、转账多少金额呢？
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
================================== Ai Message ==================================
Tool Calls:
  transfer_money (call_00_FHnIlOfcPT1aEn3JTijn8930)
 Call ID: call_00_FHnIlOfcPT1aEn3JTijn8930
  Args:
    amount: 2000
    to: 王小明
================================= Tool Message =================================
Name: transfer_money

已向王小明转账2000元
================================== Ai Message ==================================

转账已完成！已成功向王小明转账2000元。请问还有其他需要帮助的吗？


reject：reject是拒绝执行tool，我们需要把拒绝的原因告知AI

In [16]:
response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [
            {
                "type": "reject",
                # 把reject的原因告诉ai
                "message": "用户取消转账"
            }
        ]}
    ),
    config=config
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================


================================== Ai Message ==================================

您好，我是账户管理助手，可以帮助您进行转账操作。请问您需要转账给谁、转账多少金额呢？
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
================================== Ai Message ==================================
Tool Calls:
  transfer_money (call_00_FHnIlOfcPT1aEn3JTijn8930)
 Call ID: call_00_FHnIlOfcPT1aEn3JTijn8930
  Args:
    amount: 2000
    to: 王小明
================================= Tool Message =================================
Name: transfer_money

已向王小明转账2000元
================================== Ai Message ==================================

转账已完成！已成功向王小明转账2000元。请问还有其他需要帮助的吗？
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
================================== Ai Message ==================================
Tool Calls:
  transfer_

edit：edit是指需要人工对调用工具的参数做修改，所以要指定新的参数

In [18]:
response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [
            {
                "type": "edit",
                # 修改操作
                "edited_action":{
                    # 指定修改的工具
                    "name": "transfer_money",
                    # 指定修改的参数
                    "args": {"amount": 1000, "to": "王二明"}
                }
            }
        ]}
    ),
    config=config
)

for m in response["messages"]:
    print(m.pretty_print())

================================ Human Message =================================


None
================================== Ai Message ==================================

您好，我是账户管理助手，可以帮助您进行转账操作。请问您需要转账给谁、转账多少金额呢？
None
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
None
================================== Ai Message ==================================
Tool Calls:
  transfer_money (call_00_FHnIlOfcPT1aEn3JTijn8930)
 Call ID: call_00_FHnIlOfcPT1aEn3JTijn8930
  Args:
    amount: 2000
    to: 王小明
None
================================= Tool Message =================================
Name: transfer_money

已向王小明转账2000元
None
================================== Ai Message ==================================

转账已完成！已成功向王小明转账2000元。请问还有其他需要帮助的吗？
None
================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223
None
================================== Ai Message =======================

# 自定义中间件

利用LangChain提供的hook创建完全自定义的中间件。

## node-style装饰器
- Node-style hooks：在具体某个节点执行的中间件，包含：
  - before_agent
  - before_model
  - after_model
  - after_agent

函数的参数和返回值必须严格按照下面的示例

In [19]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from typing import NotRequired, Any
from langchain.agents.middleware import after_model

# 自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数

# 定义中间件
@after_model(state_schema=CustomAgentState)
def increment_counter(state: CustomAgentState, runtime: Runtime) -> dict[str, Any]:
    """使用装饰器的after_model钩子 - 统计调用计数"""
    current_count = state.get("model_call_count", 0)
    return {"model_call_count": current_count + 1}

In [20]:
agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": HumanMessage("Hello"),
        # 初始化state
        "model_call_count": 0
    },
    config=config
)

for m in response["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

Hello
None
================================== Ai Message ==================================

Hello! How can I help you today?
None


In [21]:
print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='1b97a969-06b1-4be7-a073-0783c5ff5d8f'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need respond to user. Need infer maybe wants greeting. Need answer "Hello! How can I help you today?" Keep concise.'}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 84, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 84}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '53b82a3d-874f-4526-8e81-717ed0af99b5', 'finish_reason': 'stop', 'logprobs': None},

## wrap-style装饰器

- Wrap-style hooks：环绕model或tool调用的中间件，包括：
  - wrap_model_call
  - wrap_tool_call

定义一个可以在调用模型时失败重试的中间件，最大重试次数为3次。

In [22]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Any, Callable

@wrap_model_call
def retry_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)  # 尝试调用模型
        except Exception as e:
            print(f"Retry {attempt + 1} / 3 after error: {e}")
            if attempt == 2:
                raise
    return None

In [23]:
agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[retry_call]
)

In [24]:
config = {"configurable": {"thread_id": "2"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='95266d4f-9d70-4533-9032-74a74cb19f4f'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer hello. Need maybe respond. Simple.'}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 84, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 11, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 84}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': 'af84e945-196f-4991-8030-c2cc20014c39', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fbbb0-87d8-7a71-9646-27a5b508ee51-0', tool_calls=[], 

## 类装饰器

对于需要同时用到多个hooks的更复杂的中间件逻辑，我们还可以使用自定义类继承AgentMiddleware的方式来创建中间件。

例如，一个用来记录日志的中间件，要在模型调用、工具调用前后记录日志

In [28]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import ModelCallResult, ToolCallRequest, ToolMessage
from langgraph.types import Command
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent

class LoggingMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelCallResult:
        try:
            print(f"\n========About to call model with {len(request.messages)} messages========")
            return handler(request)
        except Exception as e:
            print(f"\n========[错误]: {str(e)}========")        
            return AIMessage("调用模型失败，请重试")

    def wrap_tool_call(self, request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage | Command]) -> ToolMessage | Command:
        print(f"\n========调用工具: {request.tool_call['name']}=======")
        print(f"\n========参数: {request.tool_call['args']}=======")
        try:
            result = handler(request)
            print("\n=======工具调用成功！========")
            return result
        except Exception as e:
            print("\n=======工具调用失败: {e}========")
            raise

@tool
def get_weather(location: str):
    """查询指定城市的天气信息"""
    return f"Current weather in {location} in sunny, 25℃"

agent = create_agent(
    model = "deepseek-v4-flash",
    middleware=[LoggingMiddleware()],
    tools=[get_weather]
)

for chunk, metadata in agent.stream(
    {"messages": [HumanMessage("今天杭州天气如何？")]},
    stream_mode = "messages"
):
    if chunk and chunk.content:
        print(chunk.content, end="", flush=True)


========About to call model with 1 messages========
我来帮您查询杭州今天的天气情况。
========调用工具: get_weather=======

========参数: {'location': '杭州'}=======

=======工具调用成功！========
Current weather in 杭州 in sunny, 25℃
========About to call model with 3 messages========
根据查询结果，杭州今天的天气情况如下：

- **天气状况**：☀️ 晴天
- **气温**：25℃

今天杭州天气晴朗，气温舒适，非常适合外出活动。不过25℃对于白天来说可能稍显温暖，建议穿着轻便的衣物，并注意防晒补水哦！

# 高级用法

中间件除了利用hook做基本的信息记录和判断，还可以有一些高级的用法，例如：
- 动态修改请求：可以拦截发送给模型的请求，动态修改请求中使用的模型、工具、提示词等
- 条件跳转：在满足条件的情况下直接跳转到某个Agent执行的节点

## 动态修改请求参数

利用request.overide()来覆盖原本的request参数。

In [29]:
from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass
from langchain.chat_models import init_chat_model

reasoning_model = init_chat_model(model="deepseek-v4-pro")
chat_model = init_chat_model(model="deepseek-v4-flash")

# 定义一个context，记录用户是否使用推理模型
@dataclass
class UserContext:
    reasoning: bool = False

# 创建中间件
@wrap_model_call
def dynamic_model_selector(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """
    根据context中的reasoning来选择模型：
    - True: 选择推理模型
    - False: 选择普通对话模型
    """
    # 获取上下文参数
    is_reasoning = getattr(request.runtime.context, 'reasoning', False)

    # 根据消息数量选择模型
    selected_model = reasoning_model if is_reasoning else chat_model
    print(f"选择模型: {selected_model.model_name} (reasoning={is_reasoning})")

    # 覆盖request中的model参数
    modified_request = request.override(model=selected_model)
    return handler(modified_request)

In [31]:
agent = create_agent(
    model = "deepseek-v4-flash",
    middleware=[dynamic_model_selector],
    checkpointer=InMemorySaver(),
    context_schema=UserContext
)

config = {"configurable": {"thread_id": "2"}}

response = agent.invoke(
    {"messages": [HumanMessage("hello")]},
    config=config,
    context = UserContext(reasoning=True)  # 通过传递context控制模型的类型
)

print(response)

选择模型: deepseek-v4-pro (reasoning=True)
{'messages': [HumanMessage(content='hello', additional_kwargs={}, response_metadata={}, id='3ce8b740-e2f9-46ee-8e45-e8062c7afdb3'), AIMessage(content='Hello! 👋 How can I help you today? Feel free to ask me anything—I’m here to assist!', additional_kwargs={'refusal': None, 'reasoning_content': 'We are asked: "hello". This is just a greeting. The user might be testing the system or starting a conversation. I should respond politely and introduce myself, perhaps asking how I can help. Since I\'m expected to be a "thinking" AI, I can just reply like a friendly assistant. I\'ll keep it concise.'}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 5, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 67, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_h

## 条件跳转

在中间件使用jump_to指令可以跳过模型调用，直接进入指定的Agent节点。

模型限流的中间件，当判断用户模型调用次数超过阈值，直接禁止调用，结束Agent运行。

In [59]:
from langgraph.runtime import Runtime
from typing import NotRequired, Any
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState

# 定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agentstate，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数

# 定义中间件
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):
    def __init__(self, max_limit: int = 10):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook, 统计模型调用次数计数
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook, 校验模型调用次数是否超限
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: CustomAgentState, runtime:Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count} / {self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"模型调用次数超过最大限制：{self.max_limit}!")
            }
        return None

In [60]:
agent = create_agent(
    model = "deepseek-v4-flash",
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)

config = {"configurable": {"thread_id": "11"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {
        "messages": [HumanMessage("Hello")],
        "model_call_count": 0
    },
    config=config
)

print(response)

当前模型调用次数：1 / 2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='d35b3e04-89bb-4262-9f94-525473945fa5'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer hello. Need maybe comply. User says Hello. We respond greeting.'}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 84, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 17, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 84}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': 'ced03b54-1c38-40a2-ba7d-688ca2f603c2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fbc82-d68

In [61]:
response = agent.invoke(
    {"messages": [HumanMessage(content="讲个笑话")]},
    config=config
)

print(response)

当前模型调用次数：2 / 2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='d35b3e04-89bb-4262-9f94-525473945fa5'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer hello. Need maybe comply. User says Hello. We respond greeting.'}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 84, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 17, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 84}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': 'ced03b54-1c38-40a2-ba7d-688ca2f603c2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fbc82-d68

In [63]:
response = agent.invoke(
    {"messages": [HumanMessage(content="讲个笑话")]},
    config=config
)

for m in response["messages"]:
    print(m.pretty_print())

当前模型调用次数：3 / 2
================================ Human Message =================================

Hello
None
================================== Ai Message ==================================

Hello! How can I help you today?
None
================================ Human Message =================================

讲个笑话
None
================================== Ai Message ==================================

好的，讲一个经典冷笑话：

有一天，0和8在街上遇见，0不屑地看了8一眼，说：“胖就胖呗，还系什么腰带！”

哈哈，冷到了吗？😄
None
================================ Human Message =================================

讲个笑话
None
================================== Ai Message ==================================

模型调用次数超过最大限制：2!
None
================================ Human Message =================================

讲个笑话
None
================================== Ai Message ==================================

模型调用次数超过最大限制：2!
None
